In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Read all KPI tables
kpi_total_revenue = spark.table("novacart_dev.gold.kpi_total_revenue")
kpi_completed_orders = spark.table("novacart_dev.gold.kpi_completed_order_count")
kpi_order_rate = spark.table("novacart_dev.gold.kpi_completed_order_rate")
kpi_aov = spark.table("novacart_dev.gold.kpi_average_order_value")
kpi_active_customers = spark.table("novacart_dev.gold.kpi_active_customers_count")
kpi_dq = spark.table("novacart_dev.gold.kpi_data_quality_score")
kpi_revenue_by_country = spark.table("novacart_dev.gold.kpi_revenue_by_country")
kpi_revenue_by_channel = spark.table("novacart_dev.gold.kpi_revenue_by_channel")
kpi_top_products = spark.table("novacart_dev.gold.kpi_top_products")

# Create a unified KPI data cube with all metrics
kpi_data = []

# KPI 1: Total Revenue
total_rev = kpi_total_revenue.collect()[0]["total_revenue_usd"]
kpi_data.append(("Financial Metrics", "Total Revenue", total_rev, "USD"))

# KPI 4 & 5: Order Metrics
order_data = kpi_order_rate.collect()[0]
kpi_data.append(("Order Metrics", "Total Orders", float(order_data["total_orders"]), "Count"))
kpi_data.append(("Order Metrics", "Completed Orders", float(order_data["completed_orders"]), "Count"))
kpi_data.append(("Order Metrics", "Completion Rate", order_data["completed_order_rate"], "Percentage"))

# KPI 6: Average Order Value
aov_data = kpi_aov.collect()[0]
kpi_data.append(("Financial Metrics", "Average Order Value", aov_data["avg_order_value"], "USD"))
kpi_data.append(("Financial Metrics", "Min Order Value", aov_data["min_order_value"], "USD"))
kpi_data.append(("Financial Metrics", "Max Order Value", aov_data["max_order_value"], "USD"))

# KPI 8: Customer Metrics
cust_data = kpi_active_customers.collect()[0]
kpi_data.append(("Customer Metrics", "Total Customers", float(cust_data["total_customers"]), "Count"))
kpi_data.append(("Customer Metrics", "Active Customers", float(cust_data["active_customers"]), "Count"))
kpi_data.append(("Customer Metrics", "Customer Active Rate", cust_data["active_rate"], "Percentage"))

# KPI 10: Data Quality
dq_data = kpi_dq.collect()[0]
kpi_data.append(("Data Quality", "Total Records", float(dq_data["total_records"]), "Count"))
kpi_data.append(("Data Quality", "Clean Records", float(dq_data["clean_records"]), "Count"))
kpi_data.append(("Data Quality", "Records with Issues", float(dq_data["records_with_issues"]), "Count"))
kpi_data.append(("Data Quality", "Quality Score", dq_data["data_quality_score"], "Percentage"))

# Top Country by Revenue
top_country = kpi_revenue_by_country.orderBy(F.desc("revenue_usd")).first()
kpi_data.append(("Geographic Metrics", f"Top Country: {top_country['order_country']}", top_country["revenue_usd"], "USD"))
kpi_data.append(("Geographic Metrics", "Top Country Order Items", float(top_country["order_items"]), "Count"))

# Top Channel by Revenue
top_channel = kpi_revenue_by_channel.orderBy(F.desc("revenue_usd")).first()
kpi_data.append(("Channel Metrics", f"Top Channel: {top_channel['channel']}", top_channel["revenue_usd"], "USD"))
kpi_data.append(("Channel Metrics", "Top Channel Orders", float(top_channel["orders"]), "Count"))

# Top Product by Revenue
top_product = kpi_top_products.orderBy(F.desc("total_revenue")).first()
kpi_data.append(("Product Metrics", f"Top Product: {top_product['product_name']}", top_product["total_revenue"], "USD"))
kpi_data.append(("Product Metrics", "Top Product Units Sold", float(top_product["total_quantity_sold"]), "Count"))

# Create the data cube DataFrame
schema = StructType([
    StructField("category", StringType(), False),
    StructField("metric_name", StringType(), False),
    StructField("value", DoubleType(), False),
    StructField("unit", StringType(), False)
])

kpi_cube_df = spark.createDataFrame(kpi_data, schema)

# Add a timestamp for when this cube was generated
kpi_cube_df = kpi_cube_df.withColumn("generated_at", F.current_timestamp())

# Save the data cube as a table
kpi_cube_df.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_data_cube")

print("✓ KPI Data Cube created successfully!")
print(f"  Total metrics: {kpi_cube_df.count()}")
print("\nDisplaying complete KPI Data Cube:")
display(kpi_cube_df.orderBy("category", "metric_name"))